In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

from plotnine import *
import polars as pl

## Get significant gene trait associations

In [ ]:
# plof_df = pd.read_parquet("/s/project/geno2pheno/funcrvp/paper_results/paper_burden/at_filteredv3_pLoF_0.25_genes_NEWsplit.pq")
# plof_assoc = plof_df[plof_df.significant==True]

plof_df = pl.read_parquet("/s/project/geno2pheno/funcrvp/paper_results/paper_burden/at_filteredv3_pLoF_0.25_genes_NEWsplit.pq")
plof_assoc = plof_df.filter(plof_df['significant'] == True)
plof_assoc

## Get embedding

In [ ]:
emb = pd.read_csv('/s/project/geno2pheno/data/embeddings/pops_mat_pca256_omics.tsv', sep='\t').set_index('gene_id')
emb

## Check for one trait

In [ ]:
hdl_assocs = plof_assoc[plof_assoc.trait=='HDL_cholesterol']
hdl_assocs.head()

In [ ]:
emb_hdl = emb[emb.index.isin(hdl_assocs.gene_id)]
emb_hdl

In [ ]:
import umap

# Perform UMAP
umap_model = umap.UMAP(n_components=2, random_state=42)
umap_result = umap_model.fit_transform(emb)  # Assuming the index is 'gene_id'

# Highlight genes from the hdl_assocs dataframe
highlight_indices = emb[emb.gene_id.isin(hdl_assocs.gene_id)].index
plt.figure(figsize=(8, 6))
plt.scatter(umap_result[:, 0], umap_result[:, 1], alpha=0.1, label='Other Genes')
plt.scatter(umap_result[highlight_indices, 0], umap_result[highlight_indices, 1], alpha=0.3, color='red', label='HDL Genes')
plt.title('UMAP of Embedding with Highlighted HDL Genes')
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.legend()
plt.show()


## Compute pairwise distances

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Compute pairwise distances
pairwise_distances = squareform(pdist(emb))

pairwise_distances

In [ ]:
emb_dist = pd.DataFrame(pairwise_distances, index=emb.index, columns=emb.index)
# emb_dist.to_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/omics_pops_pairwise_distances.pq')
emb_dist

In [ ]:
emb_dist = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances.pq')
emb_dist

In [ ]:
melted_distances = emb_dist.reset_index().melt(id_vars='gene_id', var_name='gene_id_2', value_name='distance').rename(columns={'gene_id': 'gene_id_1'})
# Sort gene IDs in each row to handle switching
melted_distances['sorted_gene_ids'] = melted_distances.apply(
    lambda row: tuple(sorted([row['gene_id_1'], row['gene_id_2']])), axis=1
)

melted_distances

## Distance to nearest neighbour

In [ ]:
melted_distances = pl.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances_melt.pq')
melted_distances

In [ ]:
emb_dist_min = melted_distances.group_by('gene_id_1').agg(
    pl.col('distance').min().alias('min_distance')
    ).with_columns(
        pl.lit("all").alias('trait'),
        pl.len().alias('num_pairs')
    )
emb_dist_min

In [ ]:
emb_dist_list = []
for trait in tqdm(plof_assoc['trait'].unique()):
    trait_genes = plof_assoc.filter(plof_assoc['trait']==trait)['gene_id']

    emb_stats = melted_distances.filter(
        pl.col('gene_id_1').is_in(trait_genes) &
        pl.col('gene_id_2').is_in(trait_genes)
    ).group_by('gene_id_1').agg(
        pl.col('distance').min().alias('min_distance')
    ).with_columns(
        pl.lit(trait).alias('trait'),
        pl.len().alias('num_pairs')
    )
    
    emb_dist_list.append(emb_stats)

emb_dist_all_traits = pl.concat(emb_dist_list)

# emb_dist_all_traits = emb_dist_all_traits.group_by('trait').agg(
#     pl.len().alias('num_pairs')
# )
emb_dist_all_traits

In [ ]:
nn_dist = pl.concat([emb_dist_min, emb_dist_all_traits])
nn_dist.write_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_nn_distance.pq')

## NN plots

In [ ]:
nn_dist = pl.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_nn_distance.pq')
nn_dist

In [ ]:
stats_df = pl.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap_more_stats.pq').filter(pl.col('model') == 'funcrvp')
stats_df

In [ ]:
nn_dist = nn_dist.join(stats_df[['trait', 'comparison_bonf']], on='trait', how='left')
nn_dist

In [ ]:

plot_dt = nn_dist.with_columns(pl.concat_list([pl.col('trait'), pl.col('num_pairs').cast(pl.Utf8())]).alias("trait_label")).with_columns(pl.col("trait_label").list.join(" " ))
plot_dt = plot_dt.filter(pl.col('trait')!='all')

trait_order = plot_dt.group_by('trait_label').agg(pl.median('min_distance').alias('median_distance')).sort('median_distance')['trait_label'].to_list()
plot_dt = plot_dt.with_columns(pl.col('trait_label').cast(pl.Enum(categories=trait_order)))
plot_dt

In [ ]:
plot_dt.write_csv('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/embedding_nn_dist_stats.tsv', separator='\t')

In [ ]:
plot_dt.filter((pl.col('num_pairs')>=10) & (pl.col('num_pairs')<=30)).select(['trait', 'comparison_bonf']).unique()['comparison_bonf'].value_counts()

In [ ]:
mean_nn_dist = nn_dist.filter(pl.col('trait')=='all').select(pl.col('min_distance')).median().item()

(
    ggplot(plot_dt.filter((pl.col('num_pairs')>=10) & (pl.col('num_pairs')<=30)), aes(x='comparison_bonf', y='min_distance', color='comparison_bonf')) +
    geom_boxplot() +
    # geom_point(position=position_dodge(width=0.75)) +  # Dodge points slightly
    # geom_pointrange(aes(ymin='ymin', ymax='ymax'), position=position_dodge(width=0.75)) + # Dodge pointranges
    geom_hline(aes(yintercept=mean_nn_dist), linetype='dashed') +
    theme_classic() +
    theme(
        axis_text_x=element_text(rotation=90), # Rotate labels vertically
        figure_size=(7, 7) # Increased figure size for better label readability
    ) +
    labs(
        title="Nearest neighbor Distance for Associated Genes",
        x="",
        y="Nearest neighbour Distance",
        color="Associated Pair"
    ) +
    scale_x_discrete(name="") # Ensure x-axis label is explicitly removed if needed after categorical conversion
)

In [ ]:
mean_nn_dist = nn_dist.filter(pl.col('trait')=='all').select(pl.col('min_distance')).median().item()

(
    ggplot(plot_dt, aes(x='trait_label', y='min_distance', color='comparison_bonf')) +
    geom_boxplot() +
    # geom_point(position=position_dodge(width=0.75)) +  # Dodge points slightly
    # geom_pointrange(aes(ymin='ymin', ymax='ymax'), position=position_dodge(width=0.75)) + # Dodge pointranges
    geom_hline(aes(yintercept=mean_nn_dist), linetype='dashed') +
    theme_classic() +
    theme(
        axis_text_x=element_text(rotation=90), # Rotate labels vertically
        figure_size=(14, 7) # Increased figure size for better label readability
    ) +
    labs(
        title="Nearest neighbor Distance for Associated Genes",
        x="",
        y="Nearest neighbour Distance",
        color="Associated Pair"
    ) +
    scale_x_discrete(name="") # Ensure x-axis label is explicitly removed if needed after categorical conversion
)

## Process distances

In [ ]:
# melted_distances.to_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances_melt.pq')
emb = pl.read_csv('/s/project/geno2pheno/data/embeddings/pops_mat_pca256_omics.tsv', separator='\t')

melted_distances = pl.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances_melt.pq')

In [ ]:
emb_dist_list = []
for trait in plof_assoc['trait'].unique():
# trait = 'HDL_cholesterol'
    trait_genes = plof_assoc.filter(plof_assoc['trait']==trait)['gene_id']

    emb_stats = melted_distances.with_columns(
        associated=(
            pl.col('gene_id_1').is_in(trait_genes) &
            pl.col('gene_id_2').is_in(trait_genes)
        )
    ).group_by('associated').agg(
        [
            pl.len().alias('num_pairs'),  # Add count aggregation
            pl.col('distance').median().alias('median_distance'),
            pl.col('distance').mean().alias('mean_distance'),
            pl.col('distance').std().alias('sd_distance'),
            pl.col('distance').min().alias('min_distance'),
            pl.col('distance').max().alias('max_distance'),
        ]
    ).with_columns(
        num_genes = pl.when(pl.col('associated'))
                    .then(pl.lit(len(trait_genes)))
                    .otherwise(pl.lit(len(emb) - len(trait_genes)))
    ).with_columns(
        pl.lit(trait).alias('trait')
    )

    emb_dist_list.append(emb_stats)

In [ ]:
emb_dist_all_traits = pl.concat(emb_dist_list)
emb_dist_all_traits

In [ ]:
# Calculate overall statistics for all pairs
all_pairs_stats = melted_distances.select(
    [
        pl.len().alias('num_pairs'),
        pl.col('distance').median().alias('median_distance'),
        pl.col('distance').mean().alias('mean_distance'),
        pl.col('distance').std().alias('sd_distance'),
        pl.col('distance').min().alias('min_distance'),
        pl.col('distance').max().alias('max_distance'),
    ]
).with_columns(
    associated = pl.lit(None, dtype=pl.Boolean), # Use None for the 'associated' status
    num_genes = pl.lit(len(emb)), # Total number of genes
    trait = pl.lit('all_pairs') # Label for this row
)

# Reorder columns to match emb_dist_all_traits if necessary
all_pairs_stats = all_pairs_stats.select(emb_dist_all_traits.columns)

# Concatenate the overall stats row to the existing dataframe
emb_dist_all_traits = pl.concat([emb_dist_all_traits, all_pairs_stats])

emb_dist_all_traits

In [ ]:
emb_dist_all_traits.write_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances_assoc_stats.pq')

## Make plots

In [ ]:
plot_dt = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/embedding_distances/omics_pops_pairwise_distances_assoc_stats.pq')
plot_dt[plot_dt['trait'] == 'Mean_sphered_cell_volume']

In [ ]:
stats_df = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_models_phenopred_10k_bootstrap_more_stats.pq').query("model == 'funcrvp'")
stats_df

In [ ]:
# Get the mean distance for 'all_pairs' to draw the horizontal line
all_pairs_mean_distance = plot_dt[plot_dt['trait']=='all_pairs']['mean_distance'].item()

plot_dt['ymin'] = plot_dt['mean_distance'] - plot_dt['sd_distance']
plot_dt['ymax'] = plot_dt['mean_distance'] + plot_dt['sd_distance']

plot_dt = plot_dt.query("associated==True").merge(stats_df, on='trait').sort_values('ymax')

# Filter for associated=True and sort by mean_distance
plot_dt_assoc_true = plot_dt[plot_dt['associated'] == True]

# Create new labels with gene counts
plot_dt['trait_label'] = plot_dt.apply(lambda row: f"{row['trait']} ({row['num_genes']})", axis=1)

# Get the sorted order of trait labels
trait_order = plot_dt['trait_label'].tolist()

# Create a mapping from original trait name to the new label
trait_label_map = plot_dt.set_index('trait')['trait_label'].to_dict()

# Apply the mapping to the original dataframe to create the new label column
plot_dt['trait_label'] = plot_dt['trait'].map(trait_label_map)

# Convert the new label column to a categorical type with the specified order
# Filter out rows where trait_label might be NaN (like the 'all_pairs' row if it wasn't mapped)
plot_dt_filtered = plot_dt.dropna(subset=['trait_label'])
plot_dt_filtered['trait_label'] = pd.Categorical(plot_dt_filtered['trait_label'], categories=trait_order, ordered=True)

In [ ]:


(
    ggplot(plot_dt_filtered, aes(x='trait_label', y='mean_distance', color='comparison')) +
    # ggplot(plot_dt_filtered, aes(x='trait_label', y='ymax', color='comparison')) +
    geom_point(position=position_dodge(width=0.75)) +  # Dodge points slightly
    geom_pointrange(aes(ymin='ymin', ymax='ymax'), position=position_dodge(width=0.75)) + # Dodge pointranges
    geom_hline(aes(yintercept=all_pairs_mean_distance), linetype='dashed') +
    theme_classic() +
    theme(
        axis_text_x=element_text(rotation=90), # Rotate labels vertically
        figure_size=(14, 7) # Increased figure size for better label readability
    ) +
    labs(
        title="Mean Pairwise Embedding Distance for Associated vs Non-Associated Genes",
        x="",
        y="Mean Pairwise Distance (±SD)",
        color="Associated Pair"
    ) +
    scale_x_discrete(name="") # Ensure x-axis label is explicitly removed if needed after categorical conversion
)